In [2]:
# ================================================================
# CONUS LINE ENDPOINT SNAP + CONTINUITY WORKFLOW
#
# WORKFLOW:
#   1) Load input line dataset
#   2) Convert multipart lines to individual parts
#   3) Extract line endpoints
#   4) Find nearest endpoint for each endpoint
#   5) Identify MUTUAL nearest-endpoint pairs within tolerance
#   6) Snap paired endpoints together
#   7) Merge connected line segments
#   8) Dissolve resulting geometry
#   9) Export final continuous line dataset
#  10) Print detailed connectivity diagnostics
#
# PURPOSE:
#   Connect manually cleaned line segments where their endpoints
#   are very close to one another.
#
# IMPORTANT:
#   - Only line ENDPOINTS are considered.
#   - No interior vertices are moved.
#   - Connections are only made when endpoint pairs are mutually
#     nearest neighbors.
#   - SNAP_TOLERANCE_M controls the maximum allowed gap.
#
# ================================================================

import os
import time
import math

import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point, LineString
from shapely.strtree import STRtree
from shapely.ops import linemerge

from tqdm import tqdm


# ================================================================
# USER SETTINGS
# ================================================================

INPUT_GPKG = r"C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\line_topology\Interstates_CONUS_Final_9_30_dissolve_multipart.gpkg"
INPUT_LAYER = "Interstates_CONUS_Final_9_30_dissolve_multipart"

OUTPUT_GPKG = r"C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\line_topology\CONUS_Continuous_Lines.gpkg"
OUTPUT_LAYER = "CONUS_Continuous_Lines"

# Maximum distance allowed between endpoints
SNAP_TOLERANCE_M = 100.0

# Require the nearest endpoint relationship to be mutual
REQUIRE_MUTUAL_NEAREST = True

# ================================================================
# CONSOLE / TIMING
# ================================================================

START_TIME = time.time()


def print_banner(title):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def print_step(step, total, title):
    print()
    print(f"STEP {step}/{total}: {title}")
    print("-" * 70)


def elapsed_time():
    seconds = time.time() - START_TIME

    if seconds < 60:
        return f"{seconds:.1f} seconds"

    minutes = int(seconds // 60)
    remaining = seconds % 60

    return f"{minutes} min {remaining:.1f} sec"


# ================================================================
# START
# ================================================================

print_banner("CONUS LINE ENDPOINT SNAP + CONTINUITY WORKFLOW")

print(f"Input:              {INPUT_GPKG}")
print(f"Input Layer:        {INPUT_LAYER}")
print(f"Output:             {OUTPUT_GPKG}")
print(f"Output Layer:       {OUTPUT_LAYER}")
print(f"Snap Tolerance:     {SNAP_TOLERANCE_M:.1f} meters")
print(f"Mutual Nearest:     {REQUIRE_MUTUAL_NEAREST}")


# ================================================================
# STEP 1 — LOAD INPUT
# ================================================================

print_step(1, 9, "Loading input line dataset")

gdf = gpd.read_file(
    INPUT_GPKG,
    layer=INPUT_LAYER
)

print(f"Input features:     {len(gdf):,}")
print(f"CRS:                {gdf.crs}")

if gdf.empty:
    raise ValueError("Input dataset contains no features.")

if gdf.crs is None:
    raise ValueError(
        "Input dataset has no CRS. A projected CRS in meters is required."
    )


# ================================================================
# CHECK CRS
# ================================================================

if not gdf.crs.is_projected:
    raise ValueError(
        "Input CRS is geographic. "
        "Please project the dataset to a CRS with linear units in meters."
    )

axis_info = gdf.crs.axis_info

if axis_info:
    unit_name = axis_info[0].unit_name
    print(f"Linear unit:        {unit_name}")

    if "metre" not in unit_name.lower() and "meter" not in unit_name.lower():
        raise ValueError(
            f"Input CRS units are '{unit_name}'. "
            "This workflow requires meters."
        )


# ================================================================
# REMOVE NULL / EMPTY GEOMETRIES
# ================================================================

null_count = gdf.geometry.isna().sum()
empty_count = gdf.geometry.is_empty.sum()

if null_count > 0 or empty_count > 0:

    print()
    print("Geometry cleanup:")
    print(f"  Null geometries:  {null_count:,}")
    print(f"  Empty geometries: {empty_count:,}")

    gdf = gdf[
        gdf.geometry.notna() &
        ~gdf.geometry.is_empty
    ].copy()

    print(f"  Remaining:        {len(gdf):,}")


# ================================================================
# STEP 2 — CONVERT MULTIPART TO SINGLEPART
# ================================================================

print_step(2, 9, "Converting multipart lines to individual parts")

singlepart_records = []

for source_id, geometry in enumerate(
    tqdm(
        gdf.geometry,
        total=len(gdf),
        desc="Converting multipart lines",
        unit="feature"
    )
):

    if geometry.geom_type == "LineString":

        singlepart_records.append(
            {
                "source_id": source_id,
                "geometry": geometry
            }
        )

    elif geometry.geom_type == "MultiLineString":

        for part in geometry.geoms:

            if not part.is_empty and len(part.coords) >= 2:

                singlepart_records.append(
                    {
                        "source_id": source_id,
                        "geometry": part
                    }
                )

    else:

        raise TypeError(
            f"Unsupported geometry type: {geometry.geom_type}"
        )


lines = gpd.GeoDataFrame(
    singlepart_records,
    geometry="geometry",
    crs=gdf.crs
)

print(f"Original features:  {len(gdf):,}")
print(f"Singlepart lines:   {len(lines):,}")


# ================================================================
# STEP 3 — EXTRACT ENDPOINTS
# ================================================================

print_step(3, 9, "Extracting line endpoints")

endpoint_records = []

for line_id, geometry in enumerate(
    tqdm(
        lines.geometry,
        total=len(lines),
        desc="Extracting endpoints",
        unit="line"
    )
):

    coords = list(geometry.coords)

    start_point = Point(coords[0])
    end_point = Point(coords[-1])

    endpoint_records.append(
        {
            "endpoint_id": len(endpoint_records),
            "line_id": line_id,
            "position": "START",
            "geometry": start_point
        }
    )

    endpoint_records.append(
        {
            "endpoint_id": len(endpoint_records),
            "line_id": line_id,
            "position": "END",
            "geometry": end_point
        }
    )


endpoints = gpd.GeoDataFrame(
    endpoint_records,
    geometry="geometry",
    crs=gdf.crs
)

print(f"Line segments:      {len(lines):,}")
print(f"Endpoints:          {len(endpoints):,}")


# ================================================================
# STEP 4 — BUILD ENDPOINT SPATIAL INDEX
# ================================================================

print_step(4, 9, "Building endpoint spatial index")

endpoint_geometries = endpoints.geometry.tolist()

endpoint_tree = STRtree(endpoint_geometries)

print(f"Indexed endpoints:  {len(endpoint_geometries):,}")
print(f"Search tolerance:    {SNAP_TOLERANCE_M:.1f} m")


# ================================================================
# STEP 5 — FIND MUTUAL NEAREST ENDPOINTS
# ================================================================

print_step(5, 9, "Finding nearest endpoint pairs")

nearest_endpoint = {}
nearest_distance = {}

for endpoint_id, point in enumerate(
    tqdm(
        endpoint_geometries,
        total=len(endpoint_geometries),
        desc="Searching endpoint neighbors",
        unit="endpoint"
    )
):

    # Search all endpoints within tolerance
    candidate_indices = endpoint_tree.query(
        point.buffer(SNAP_TOLERANCE_M)
    )

    best_id = None
    best_distance = float("inf")

    for candidate_id in candidate_indices:

        candidate_id = int(candidate_id)

        # Never match an endpoint to itself
        if candidate_id == endpoint_id:
            continue

        # Never connect endpoints belonging to the same line
        if (
            endpoints.iloc[endpoint_id]["line_id"]
            ==
            endpoints.iloc[candidate_id]["line_id"]
        ):
            continue

        candidate_point = endpoint_geometries[candidate_id]

        distance = point.distance(candidate_point)

        if distance <= SNAP_TOLERANCE_M:

            if distance < best_distance:

                best_distance = distance
                best_id = candidate_id

    nearest_endpoint[endpoint_id] = best_id
    nearest_distance[endpoint_id] = best_distance


# ================================================================
# IDENTIFY VALID CONNECTIONS
# ================================================================

connection_pairs = []

for endpoint_id, candidate_id in nearest_endpoint.items():

    if candidate_id is None:
        continue

    if REQUIRE_MUTUAL_NEAREST:

        if nearest_endpoint.get(candidate_id) != endpoint_id:
            continue

    pair = tuple(
        sorted(
            [
                endpoint_id,
                candidate_id
            ]
        )
    )

    if pair not in connection_pairs:

        connection_pairs.append(pair)


# ================================================================
# CONNECTION DIAGNOSTICS
# ================================================================

connection_distances = []

for endpoint_a, endpoint_b in connection_pairs:

    distance = endpoint_geometries[endpoint_a].distance(
        endpoint_geometries[endpoint_b]
    )

    connection_distances.append(distance)


print()
print("Endpoint matching results:")
print(f"  Total endpoints:          {len(endpoints):,}")
print(f"  Candidate connections:    {len(connection_pairs):,}")

if connection_distances:

    print(
        f"  Minimum gap:              "
        f"{min(connection_distances):.3f} m"
    )

    print(
        f"  Maximum gap:              "
        f"{max(connection_distances):.3f} m"
    )

    print(
        f"  Mean gap:                 "
        f"{np.mean(connection_distances):.3f} m"
    )

    print(
        f"  Median gap:               "
        f"{np.median(connection_distances):.3f} m"
    )

else:

    print("  No endpoint connections found.")


# ================================================================
# STEP 6 — SNAP ENDPOINTS
# ================================================================

print_step(6, 9, "Snapping matched endpoints")

# ------------------------------------------------
# Build endpoint -> snapped coordinate dictionary
# ------------------------------------------------

snap_coordinates = {}

for endpoint_a, endpoint_b in connection_pairs:

    point_a = endpoint_geometries[endpoint_a]
    point_b = endpoint_geometries[endpoint_b]

    # Use the midpoint so neither endpoint gets
    # preferential treatment.
    midpoint = Point(
        (
            (point_a.x + point_b.x) / 2.0,
            (point_a.y + point_b.y) / 2.0
        )
    )

    snap_coordinates[endpoint_a] = midpoint
    snap_coordinates[endpoint_b] = midpoint


# ------------------------------------------------
# Rebuild lines with snapped endpoints
# ------------------------------------------------

snapped_geometries = []

for line_id, geometry in enumerate(
    tqdm(
        lines.geometry,
        total=len(lines),
        desc="Snapping line endpoints",
        unit="line"
    )
):

    coords = list(geometry.coords)

    # Find START endpoint ID
    start_endpoint_id = line_id * 2

    # Find END endpoint ID
    end_endpoint_id = line_id * 2 + 1

    # Replace start coordinate if it was matched
    if start_endpoint_id in snap_coordinates:

        snapped_start = snap_coordinates[start_endpoint_id]

        coords[0] = (
            snapped_start.x,
            snapped_start.y
        )

    # Replace end coordinate if it was matched
    if end_endpoint_id in snap_coordinates:

        snapped_end = snap_coordinates[end_endpoint_id]

        coords[-1] = (
            snapped_end.x,
            snapped_end.y
        )

    snapped_geometries.append(
        LineString(coords)
    )


snapped_lines = lines.copy()

snapped_lines.geometry = snapped_geometries

print(
    f"Endpoints snapped: {len(snap_coordinates):,}"
)

print(
    f"Endpoint pairs:     {len(connection_pairs):,}"
)


# ================================================================
# STEP 7 — MERGE CONNECTED LINEWORK
# ================================================================

print_step(7, 9, "Merging connected line segments")

merged_geometry = linemerge(
    list(snapped_lines.geometry)
)

# ------------------------------------------------
# Count resulting geometries
# ------------------------------------------------

if merged_geometry.geom_type == "LineString":

    merged_geometries = [merged_geometry]

elif merged_geometry.geom_type == "MultiLineString":

    merged_geometries = list(merged_geometry.geoms)

else:

    raise TypeError(
        f"Unexpected merged geometry type: "
        f"{merged_geometry.geom_type}"
    )


print(
    f"Connected line features: {len(merged_geometries):,}"
)


# ================================================================
# STEP 8 — CREATE FINAL OUTPUT
# ================================================================

print_step(8, 9, "Creating final continuous line dataset")

output = gpd.GeoDataFrame(
    {
        "CONTINUOUS_ID": range(
            1,
            len(merged_geometries) + 1
        )
    },
    geometry=merged_geometries,
    crs=gdf.crs
)

# ------------------------------------------------
# Remove zero-length geometries
# ------------------------------------------------

output = output[
    output.geometry.length > 0
].copy()

print(
    f"Final output features: {len(output):,}"
)


# ================================================================
# STEP 9 — EXPORT
# ================================================================

print_step(9, 9, "Exporting final dataset")

output_directory = os.path.dirname(OUTPUT_GPKG)

if output_directory:
    os.makedirs(
        output_directory,
        exist_ok=True
    )

# Remove existing layer if necessary
if os.path.exists(OUTPUT_GPKG):

    import fiona

    existing_layers = fiona.listlayers(
        OUTPUT_GPKG
    )

    if OUTPUT_LAYER in existing_layers:

        print(
            f"Removing existing layer: {OUTPUT_LAYER}"
        )

        import sqlite3

        connection = sqlite3.connect(
            OUTPUT_GPKG
        )

        cursor = connection.cursor()

        cursor.execute(
            f'DROP TABLE IF EXISTS "{OUTPUT_LAYER}"'
        )

        connection.commit()
        connection.close()


output.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG"
)


# ================================================================
# FINAL DIAGNOSTICS
# ================================================================

print_banner("FINAL RESULTS")

print(f"Original features:          {len(gdf):,}")
print(f"Singlepart segments:        {len(lines):,}")
print(f"Total endpoints:            {len(endpoints):,}")
print(f"Endpoint pairs connected:   {len(connection_pairs):,}")
print(f"Endpoints snapped:          {len(snap_coordinates):,}")
print(f"Final continuous features:  {len(output):,}")

if connection_distances:

    print()
    print("SNAP DISTANCE SUMMARY")
    print("-" * 70)

    print(
        f"Minimum:                    "
        f"{min(connection_distances):.3f} m"
    )

    print(
        f"Maximum:                    "
        f"{max(connection_distances):.3f} m"
    )

    print(
        f"Mean:                       "
        f"{np.mean(connection_distances):.3f} m"
    )

    print(
        f"Median:                     "
        f"{np.median(connection_distances):.3f} m"
    )

# ------------------------------------------------
# FINAL CONNECTIVITY CHECK
# ------------------------------------------------

print()
print("CONNECTIVITY CHECK")
print("-" * 70)

if len(output) == 1:

    print(
        "SUCCESS: All line segments were merged "
        "into ONE continuous feature."
    )

else:

    print(
        f"WARNING: Output contains {len(output):,} "
        f"separate continuous features."
    )

    print(
        "This indicates that some line endpoints "
        "remain disconnected."
    )


print()
print(f"Output: {OUTPUT_GPKG}")
print(f"Layer:  {OUTPUT_LAYER}")
print(f"Elapsed time: {elapsed_time()}")

print()
print("=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)


CONUS LINE ENDPOINT SNAP + CONTINUITY WORKFLOW
Input:              C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\line_topology\Interstates_CONUS_Final_9_30_dissolve_multipart.gpkg
Input Layer:        Interstates_CONUS_Final_9_30_dissolve_multipart
Output:             C:\Users\KyleSteen.AzureAD\Documents\ROW_Width_workspace\line_topology\CONUS_Continuous_Lines.gpkg
Output Layer:       CONUS_Continuous_Lines
Snap Tolerance:     100.0 meters
Mutual Nearest:     True

STEP 1/9: Loading input line dataset
----------------------------------------------------------------------
Input features:     637
CRS:                EPSG:5070
Linear unit:        metre

STEP 2/9: Converting multipart lines to individual parts
----------------------------------------------------------------------


Converting multipart lines: 100%|████████████████████████████████████████████| 637/637 [00:00<00:00, 42826.46feature/s]


Original features:  637
Singlepart lines:   637

STEP 3/9: Extracting line endpoints
----------------------------------------------------------------------


Extracting endpoints: 100%|██████████████████████████████████████████████████████| 637/637 [00:00<00:00, 2363.67line/s]


Line segments:      637
Endpoints:          1,274

STEP 4/9: Building endpoint spatial index
----------------------------------------------------------------------
Indexed endpoints:  1,274
Search tolerance:    100.0 m

STEP 5/9: Finding nearest endpoint pairs
----------------------------------------------------------------------


Searching endpoint neighbors: 100%|███████████████████████████████████████| 1274/1274 [00:00<00:00, 10687.21endpoint/s]



Endpoint matching results:
  Total endpoints:          1,274
  Candidate connections:    307
  Minimum gap:              0.000 m
  Maximum gap:              80.380 m
  Mean gap:                 2.476 m
  Median gap:               0.280 m

STEP 6/9: Snapping matched endpoints
----------------------------------------------------------------------


Snapping line endpoints: 100%|████████████████████████████████████████████████████| 637/637 [00:01<00:00, 632.54line/s]

Endpoints snapped: 614
Endpoint pairs:     307

STEP 7/9: Merging connected line segments
----------------------------------------------------------------------
Connected line features: 340

STEP 8/9: Creating final continuous line dataset
----------------------------------------------------------------------
Final output features: 340

STEP 9/9: Exporting final dataset
----------------------------------------------------------------------

FINAL RESULTS
Original features:          637
Singlepart segments:        637
Total endpoints:            1,274
Endpoint pairs connected:   307
Endpoints snapped:          614
Final continuous features:  340

SNAP DISTANCE SUMMARY
----------------------------------------------------------------------
Minimum:                    0.000 m
Maximum:                    80.380 m
Mean:                       2.476 m
Median:                     0.280 m

CONNECTIVITY CHECK
----------------------------------------------------------------------
This indicates th